# Tests: `fasterai.core.criteria` (source `nbs/core/criteria.ipynb`)

In [ ]:
from fastcore.test import *
import torch
import torch.nn as nn
from torch.nn.utils import parametrize
from fasterai.core.criteria import *
from fasterai.core.parametrize import _master

In [ ]:
from fastcore.test import *

# Reducer produces different results for sum vs mean on non-singleton dims
w = torch.randn(16, 3, 3, 3)
# dim=[0] is a no-op after [None] expansion (singleton), so use dim=[1]
test_ne(Reducer.sum(w, dim=[1]), Reducer.mean(w, dim=[1]))

# Normalizer.standardization → values in [0, 1] range (min-max normalization)
scores = torch.randn(100).abs()
normed = Normalizer.standardization(scores)
test_close(normed.min().item(), 0.0, eps=1e-5)
test_close(normed.max().item(), 1.0, eps=1e-5)

# Normalizer.max → max value is 1.0
max_normed = Normalizer.max(torch.randn(100).abs())
test_close(max_normed.max().item(), 1.0, eps=1e-6)
assert max_normed.min() >= 0.0

# Normalizer.gaussian → approximately zero mean
gauss = Normalizer.gaussian(torch.randn(1000))
test_close(gauss.mean().item(), 0.0, eps=0.1)

# large_final returns scores with correct shape
conv = nn.Conv2d(3, 16, 3)
scores = large_final(conv, 'weight')
test_eq(scores.shape, conv.weight.shape)

# random produces different scores each call
test_ne(random(conv, 'weight'), random(conv, 'weight'))

# Criteria with both needs_init and needs_update raises AssertionError
with ExceptionExpected(AssertionError):
    Criteria(torch.abs, needs_init=True, needs_update=True)

# grad_crit returns tensor of correct shape
conv2 = nn.Conv2d(3, 16, 3)
grad_scores = grad_crit(conv2, 'weight')
test_eq(grad_scores.shape, conv2.weight.shape)

# available_criterias returns tuple of strings
crit_list = available_criterias()
assert 'large_final' in crit_list
assert 'random' in crit_list
assert len(crit_list) > 0

In [ ]:
# --- Criteria.scale tests ---
_conv = nn.Conv2d(3, 16, 3)
_norms = torch.rand(3)  # per-input-channel norms
_scaled = Criteria(torch.abs, scale={_conv: _norms})

# Returns correct shape
test_eq(_scaled(_conv, 'weight').shape, _conv.weight.shape)

# Scaled scores differ from unscaled
test_ne(_scaled(_conv, 'weight'), large_final(_conv, 'weight'))

# --- activation_criteria / wanda tests ---
# wanda has needs_data=True
assert wanda.needs_data == True
assert large_final.needs_data == False

# wanda.calibrate populates scale
_model = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 32, 3, padding=1))
_data = [torch.randn(4, 3, 8, 8)]
wanda.calibrate(_model, _data, nn.Conv2d, n_batches=1)
assert wanda.scale is not None
test_eq(len(wanda.scale), 2)  # 2 Conv2d layers

# After calibration, returns correct shape
test_eq(wanda(_model[0], 'weight').shape, _model[0].weight.shape)

# Non-negative scores
assert wanda(_model[0], 'weight').min() >= 0

# Zero-activation channel → low score for connected weights
_zero_data = torch.zeros(4, 3, 8, 8)
_zero_data[:, 0] = 1.0  # only channel 0 is active
_wanda_zero = activation_criteria(torch.abs)
_wanda_zero.calibrate(_model, [_zero_data], nn.Conv2d, n_batches=1)
_scores = _wanda_zero(_model[0], 'weight')
assert _scores[:, 1:].max() < _scores[:, 0].mean(), "Zero-activation channels should score lower"

In [ ]:
# --- a parametrized weight: the scores are the MASTER's, not those of the weight computed from it ---
# At 4 bits a row holds at most 16 distinct magnitudes, so a threshold on a rounded score lands on a
# heavily populated level and the achieved sparsity becomes a function of the tie population.
class _Double(nn.Module):
    "A parametrization with a visible effect: the weight this module computes is twice its master"
    def forward(self, w): return w * 2

_c = nn.Conv2d(3, 16, 3)
_plain_final = large_final(_c, 'weight').clone()
_plain_grad = grad_crit(_c, 'weight').clone()
parametrize.register_parametrization(_c, 'weight', _Double())
test_eq(torch.equal(_c.weight.detach(), 2 * _master(_c).detach()), True)   # the fixture is not degenerate
test_eq(torch.equal(large_final(_c, 'weight'), _plain_final), True)
test_eq(torch.equal(grad_crit(_c, 'weight'), _plain_grad), True)

# `update_weights` snapshots the master too: an updating criteria comparing the master against a
# ROUNDED snapshot would be measuring the rounding, not the training
_c._init_weights = _master(_c).detach().clone()
updating_movement(_c, 'weight')          # registers `_old_weights` from `_init_weights`
updating_movement.update_weights(_c)
test_eq(torch.equal(_c._old_weights, _master(_c).detach()), True)
test_ne(_c._old_weights.tolist(), _c.weight.detach().tolist())

# ...so what the score measures is a step of the master
with torch.no_grad(): _master(_c).add_(0.25)
_moved = updating_movement(_c, 'weight')
test_close(float(_moved.mean()), 0.25, eps=1e-5)

In [ ]:
#| slow
# Integration test: Wanda on a real ResNet-18 with Sparsifier
from torchvision.models import resnet18
from fasterai.sparse.sparsifier import Sparsifier

_resnet = resnet18(weights=None).eval()
_cal_data = [torch.randn(4, 3, 32, 32) for _ in range(3)]

# Sparsify to 50% with Wanda
_wanda_crit = activation_criteria(torch.abs)
_sp = Sparsifier(_resnet, 'weight', 'local', _wanda_crit, data=_cal_data)
_sp.sparsify_model(0.5)

# Model still produces valid outputs
_out = _resnet(torch.randn(1, 3, 32, 32))
assert torch.isfinite(_out).all(), "Wanda-sparsified ResNet produced non-finite outputs"

# Actual sparsity close to 50%
_zeros = sum((m.weight==0).sum().item() for m in _resnet.modules() if isinstance(m, nn.Conv2d))
_total = sum(m.weight.numel() for m in _resnet.modules() if isinstance(m, nn.Conv2d))
test_close(100*_zeros/_total, 50.0, eps=5.0)